# 2WikiMultiHopQA Benchmark

This notebook benchmarks RAG methods on **2WikiMultiHopQA**, a multi-hop QA dataset
that requires reasoning across two Wikipedia articles.

**Dataset**: [2WikiMultiHopQA](https://github.com/Alab-NII/2wikimultihop) — downloaded from HuggingFace (`xanhho/2WikiMultihopQA`).
Questions involve comparison, bridge, and inference reasoning types.

**Metrics**:
- **RAGAS** (4 metrics) — retrieval and generation quality
- **Exact Match (EM)** — strict string match after normalization
- **F1** — token-level overlap (well-suited for short extractive answers)

In [ ]:
# --- Environment setup (must run first) ---
from nb_helpers.config import setup_environment, load_config
setup_environment()

from helpers.types import BenchmarkConfig

# --- Configuration ---
METHODS = ["vdb"]                  # Adapter methods to benchmark
MAX_QUESTIONS = 20                  # Number of test questions (0 = all)
TOP_K = 8                           # Chunks to retrieve per query
CHUNK_SIZE = 1200                   # Characters per chunk
CHUNK_OVERLAP = 200                 # Overlap between chunks
TWOWIKI_SPLIT = "validation"        # HuggingFace split
SKIP_INDEX = False                  # Set True to reuse existing indexes

config_dict = load_config(overrides={
    "methods": METHODS,
    "max_questions": MAX_QUESTIONS,
    "top_k": TOP_K,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
})
config = BenchmarkConfig.from_dict(config_dict)

BENCHMARK = "twowiki"
RUN_CONFIG = {
    "corpus": "2wikimultihopqa",
    "split": TWOWIKI_SPLIT,
    "num_questions": MAX_QUESTIONS,
    "top_k": TOP_K,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
}
print(f"Config: {len(METHODS)} methods, {MAX_QUESTIONS} questions, top_k={TOP_K}")

In [ ]:
# --- Load Dataset ---
from nb_helpers.datasets import load_twowiki_multihop

chunks, testset = load_twowiki_multihop(
    split=TWOWIKI_SPLIT,
    max_questions=MAX_QUESTIONS,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
print(f"\nSample question: {testset[0]['question']}")
print(f"Ground truth: {testset[0]['ground_truth']}")
print(f"Type: {testset[0].get('question_type', 'N/A')}")

In [ ]:
# --- Run Benchmarks ---
from nb_helpers.pipeline import run_all_methods, save_results

all_results = await run_all_methods(
    methods=METHODS,
    chunks=chunks,
    testset=testset,
    config=config,
    skip_index=SKIP_INDEX,
)
save_results(all_results, BENCHMARK, RUN_CONFIG)

In [ ]:
# --- Evaluate with RAGAS + EM/F1 ---
from nb_helpers.metrics import evaluate_all_methods

all_results = await evaluate_all_methods(
    all_results,
    config=config,
    use_ragas=True,
    use_em_f1=True,
)
save_results(all_results, BENCHMARK, RUN_CONFIG)

In [ ]:
# --- Summary Table ---
from nb_helpers.charts import summary_table

summary_table(all_results, extra_metrics=["exact_match", "f1"])

In [ ]:
# --- Charts ---
from nb_helpers.charts import ragas_bar_chart, latency_chart, cost_breakdown_chart, em_f1_chart

ragas_bar_chart(all_results, title="RAGAS Scores — 2WikiMultiHopQA")
em_f1_chart(all_results, title="EM & F1 — 2WikiMultiHopQA")
latency_chart(all_results, title="Avg Query Latency — 2WikiMultiHopQA")
cost_breakdown_chart(all_results, title="Cost Breakdown — 2WikiMultiHopQA")